In [1]:
# Block 1 - Imports
import os
import shutil
import matplotlib.pyplot as plt
import cv2
import random
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Image, display
from ultralytics import YOLO

ModuleNotFoundError: No module named 'cv2'

only importing libraries


In [ ]:
import os
import shutil
import kagglehub

# Download dataset

path = kagglehub.dataset_download("sukmaadhiwijaya/welding-defect-object-detection")

print("Downloaded to:", path)

# Destination (current folder)

dest = os.path.join(os.getcwd(), "dataset")


c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloaded to: C:\Users\Asus\.cache\kagglehub\datasets\sukmaadhiwijaya\welding-defect-object-detection\versions\2


dataset import 

In [ ]:
import os

# Block 3 - Define paths (FIXED - one level deeper)
v1_dir = r"C:\Users\Asus\Desktop\WELDING-DEFECT\The Welding Defect Dataset\The Welding Defect Dataset"
v2_dir = r"C:\Users\Asus\Desktop\WELDING-DEFECT\The Welding Defect Dataset - v2\The Welding Defect Dataset - v2"
merged_dir = r"C:\Users\Asus\Desktop\WELDING-DEFECT\merged"

# Verify the fix
for label, path in [('V1', v1_dir), ('V2', v2_dir)]:
    if os.path.exists(path):
        contents = os.listdir(path)
        print(f"✅ {label}: {contents}")
    else:
        print(f"❌ {label}: NOT FOUND → {path}")

✅ V1: ['data.yaml', 'test', 'train', 'valid']
✅ V2: ['data.yaml', 'test', 'train', 'valid']


In [ ]:
import os
import shutil

# Block 4 - Merge both datasets into one
for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(merged_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(merged_dir, split, 'labels'), exist_ok=True)

    for src, prefix in [(v1_dir, 'v1_'), (v2_dir, 'v2_')]:  # ✅ Prefix tied to source
        for folder in ['images', 'labels']:
            src_path = os.path.join(src, split, folder)
            dst_path = os.path.join(merged_dir, split, folder)

            if not os.path.exists(src_path):
                print(f"[SKIP] Not found: {src_path}")
                continue

            files = os.listdir(src_path)
            print(f"[COPY] {len(files)} files from {src_path}")

            for f in files:
                shutil.copy2(os.path.join(src_path, f),
                             os.path.join(dst_path, prefix + f))

print("\nDatasets merged successfully")

# Verification
for split in ['train', 'valid', 'test']:
    imgs = len(os.listdir(os.path.join(merged_dir, split, 'images')))
    lbls = len(os.listdir(os.path.join(merged_dir, split, 'labels')))
    print(f"{split}: {imgs} images, {lbls} labels")

[COPY] 839 files from C:\Users\Asus\Desktop\WELDING-DEFECT\The Welding Defect Dataset\The Welding Defect Dataset\train\images
[COPY] 839 files from C:\Users\Asus\Desktop\WELDING-DEFECT\The Welding Defect Dataset\The Welding Defect Dataset\train\labels
[COPY] 1619 files from C:\Users\Asus\Desktop\WELDING-DEFECT\The Welding Defect Dataset - v2\The Welding Defect Dataset - v2\train\images
[COPY] 1619 files from C:\Users\Asus\Desktop\WELDING-DEFECT\The Welding Defect Dataset - v2\The Welding Defect Dataset - v2\train\labels
[COPY] 176 files from C:\Users\Asus\Desktop\WELDING-DEFECT\The Welding Defect Dataset\The Welding Defect Dataset\valid\images
[COPY] 176 files from C:\Users\Asus\Desktop\WELDING-DEFECT\The Welding Defect Dataset\The Welding Defect Dataset\valid\labels
[COPY] 283 files from C:\Users\Asus\Desktop\WELDING-DEFECT\The Welding Defect Dataset - v2\The Welding Defect Dataset - v2\valid\images
[COPY] 283 files from C:\Users\Asus\Desktop\WELDING-DEFECT\The Welding Defect Dataset 

In [ ]:
import os

# Block 5 - Create merged data.yaml
# Convert to forward slashes — YOLO requires this even on Windows
merged_dir_posix = merged_dir.replace('\\', '/')

yaml_content = f"""train: {merged_dir_posix}/train/images
val: {merged_dir_posix}/valid/images
test: {merged_dir_posix}/test/images

nc: 4
names: ['crack', 'excess_reinforcement', 'porosity', 'spatters']
"""

yaml_path = os.path.join(merged_dir, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"data.yaml created at: {yaml_path}")
print(yaml_content)

# Verify the file was written correctly
with open(yaml_path, 'r') as f:
    print("Verified contents:\n", f.read())

data.yaml created at: C:\Users\Asus\Desktop\WELDING-DEFECT\merged\data.yaml
train: C:/Users/Asus/Desktop/WELDING-DEFECT/merged/train/images
val: C:/Users/Asus/Desktop/WELDING-DEFECT/merged/valid/images
test: C:/Users/Asus/Desktop/WELDING-DEFECT/merged/test/images

nc: 4
names: ['crack', 'excess_reinforcement', 'porosity', 'spatters']

Verified contents:
 train: C:/Users/Asus/Desktop/WELDING-DEFECT/merged/train/images
val: C:/Users/Asus/Desktop/WELDING-DEFECT/merged/valid/images
test: C:/Users/Asus/Desktop/WELDING-DEFECT/merged/test/images

nc: 4
names: ['crack', 'excess_reinforcement', 'porosity', 'spatters']



In [ ]:
import os
import random
import cv2
import matplotlib
matplotlib.use('Agg')  # ✅ No GUI — just saves to file, never crashes
import matplotlib.pyplot as plt

# Block 6 - Display sample training images
def display_images(image_dir, num_images=9):
    if not os.path.exists(image_dir):
        print(f"[ERROR] Directory not found: {image_dir}")
        return

    images = [f for f in os.listdir(image_dir)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    if not images:
        print(f"[ERROR] No images found in: {image_dir}")
        return

    print(f"Found {len(images)} images. Displaying {min(num_images, len(images))}...")
    sample_images = random.sample(images, min(num_images, len(images)))

    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    axes = axes.flatten()

    plotted = 0
    for image_name in sample_images:
        if plotted >= 9:
            break
        image_path = os.path.join(image_dir, image_name)
        img = cv2.imread(image_path)
        if img is None:
            print(f"[WARN] Could not read: {image_name}")
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[plotted].imshow(img_rgb)
        axes[plotted].set_title(image_name[:20], fontsize=8)
        axes[plotted].axis('off')
        plotted += 1

    # Hide unused subplots
    for i in range(plotted, 9):
        axes[i].axis('off')

    plt.tight_layout()

    save_path = r"C:\Users\Asus\Desktop\WELDING-DEFECT\sample_images.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved to: {save_path}")
    print("Open the file to view your sample images.")

merged_dir = r"C:\Users\Asus\Desktop\WELDING-DEFECT\merged"

print("Sample merged training images:")
display_images(os.path.join(merged_dir, 'train', 'images'))

Sample merged training images:
Found 2458 images. Displaying 9...
✅ Saved to: C:\Users\Asus\Desktop\WELDING-DEFECT\sample_images.png
Open the file to view your sample images.


In [ ]:
# Block 7 - EDA: class distribution
def load_labels(label_dir):
    label_files = os.listdir(label_dir)
    data = []
    for label_file in label_files:
        with open(os.path.join(label_dir, label_file), 'r') as file:
            lines = file.readlines()
            for line in lines:
                class_id, x_center, y_center, width, height = map(float, line.strip().split())
                data.append([label_file, class_id, x_center, y_center, width, height])
    df = pd.DataFrame(data, columns=['file', 'class_id', 'x_center', 'y_center', 'width', 'height'])
    return df

merged_labels = load_labels(os.path.join(merged_dir, 'train/labels'))

plt.figure(figsize=(12, 6))
sns.countplot(data=merged_labels, x='class_id')
plt.title('Class Distribution in Merged Training Set')
plt.xticks(ticks=[0,1,2,3], labels=['crack', 'excess_reinforcement', 'porosity', 'spatters'])
plt.show()

C:\Users\Asus\AppData\Local\Temp\ipykernel_29892\2854503321.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# EDA and Preprocessing
def load_labels(label_dir):
    label_files = os.listdir(label_dir)
    data = []
    for label_file in label_files:
        with open(os.path.join(label_dir, label_file), 'r') as file:
            lines = file.readlines()
            for line in lines:
                class_id, x_center, y_center, width, height = map(float, line.strip().split())
                data.append([label_file, class_id, x_center, y_center, width, height])
    df = pd.DataFrame(data, columns=['file', 'class_id', 'x_center', 'y_center', 'width', 'height'])
    return df

v1_train_labels = load_labels(os.path.join(v1_dir, 'train/labels'))
v2_train_labels = load_labels(os.path.join(v2_dir, 'train/labels'))

In [ ]:
import os
import shutil
from ultralytics import YOLO

# Block 8 - Train model
if os.path.exists('runs'):
    shutil.rmtree('runs')
    print("Cleared old runs")

# Fix yaml path to use forward slashes
yaml_path = os.path.join(merged_dir, 'data.yaml').replace('\\', '/')
print(f"Using yaml: {yaml_path}")

# Verify yaml exists before training
if not os.path.exists(yaml_path):
    print(f"❌ data.yaml not found at: {yaml_path}")
else:
    print("✅ data.yaml found, starting training...")

    model = YOLO('yolov8s.pt')
    model.train(
        data=yaml_path,
        epochs=100,
        imgsz=416,
        batch=4,
        workers=0,        # ✅ 0 instead of 2 — fixes Windows multiprocessing crash
        cache=False,
        device=0,
        patience=20,
        lr0=0.01,
        lrf=0.001,
        momentum=0.937,
        weight_decay=0.0005,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        flipud=0.3,
        fliplr=0.5,
        mosaic=1.0,
        translate=0.1,
        scale=0.5,
        project='runs/welding',
        name='v1',
        exist_ok=True     # ✅ Prevents crash if runs/welding/v1 already exists
    )

    print("✅ Training complete")
    weights_path = os.path.join('runs', 'welding', 'v1', 'weights', 'best.pt')
    if os.path.exists(weights_path):
        print(f"✅ Best weights saved at: {weights_path}")
    else:
        print("⚠️ Training finished but best.pt not found — check for early stopping")

Using yaml: C:/Users/Asus/Desktop/WELDING-DEFECT/merged/data.yaml
✅ data.yaml found, starting training...
Ultralytics 8.4.37  Python-3.11.0 torch-2.11.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: 0
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [ ]:
# Run this to find the actual weights path
import os

for root, dirs, files in os.walk('runs'):
    for file in files:
        if file.endswith('.pt'):
            print(os.path.join(root, file))

In [ ]:
# Block 9 - Load best weights and run inference
class_names = {0: 'crack', 1: 'excess_reinforcement', 2: 'porosity', 3: 'spatters'}

best_model = YOLO('runs/detect/runs/welding/v1/weights/best.pt')

def display_results(model, img_dir, num_images=5):
    images = [f for f in os.listdir(img_dir)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    if len(images) == 0:
        print(f"No images found in {img_dir}")
        return

    sample_images = random.sample(images, min(num_images, len(images)))

    for image_name in sample_images:
        image_path = os.path.join(img_dir, image_name)
        results = model.predict(source=image_path, conf=0.25, verbose=False)

        for result in results:
            result.names = class_names
            annotated = result.plot()
            annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(10, 10))
            plt.imshow(annotated_rgb)
            plt.axis('off')
            plt.title(image_name)
            plt.show()
            plt.close()

print("Results on merged test set:")
display_results(best_model, os.path.join(merged_dir, 'test/images'))